# ECG画像分類：Google Colab 推論例

Takotsubo（`tako`）と acute anterior STEMI（`AMI`）を分類する、研究用の推論例です。ローカルCLIと同じコードを呼び出します。

既定値は **ViT: TTA 24 / magnitude 1、ConvNeXtV2: TTAなし、MLP-Mixer: TTA 32 / magnitude 3**。アンサンブルは各モデルの **raw logitsを等重みで平均した後、softmax** を適用します。TTA回数には変換なしの1回を含みます。設定は下のセルで変更できます。

学習済み重み・実患者画像はこのnotebookに含まれていません。推論用に整理した重みと、Colabでの利用が認められた画像を用意してください。公開用の患者画像は不要です。既存の論文予測との一致確認には、対応する非公開画像・予測結果を別途使用します。

## 1. 使用するコードの版を指定する

初回は `GIT_REF = "main"` を使用できます。再現用には、確認済みのタグまたはcommit SHAに変更してください。チェックアウトしたSHAを記録します。

このnotebookは新しい推論コードと一緒に使用します。`v0.1.1`にはこのCLIは含まれていません。

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import json
import re
import subprocess
import sys

REPO_URL = "https://github.com/ekagawa007/takotsubo_ami_ai_ecg.git"
GIT_REF = "main"  # 確認後はcommit SHAまたは対応するReleaseタグに固定
REPO_DIR = Path("/content/takotsubo_ami_ai_ecg")

if not re.fullmatch(r"[A-Za-z0-9_][A-Za-z0-9_./-]*", GIT_REF):
    raise ValueError("GIT_REFにブランチ名、タグ名、またはcommit SHAを指定してください。")

def git(*args):
    return subprocess.check_output(
        ["git", "-C", str(REPO_DIR), *args], text=True
    ).strip()

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError("REPO_DIRが既存のGit作業フォルダではありません。別の保存先を指定してください。")
    if git("remote", "get-url", "origin").rstrip("/") != REPO_URL.rstrip("/"):
        raise RuntimeError("REPO_DIRのリポジトリがREPO_URLと一致しません。")
    if git("status", "--porcelain"):
        raise RuntimeError("作業フォルダに変更があります。保存してから再実行してください。")

subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth=1", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
CODE_COMMIT = git("rev-parse", "HEAD")
required_code = [
    REPO_DIR / "examples" / "predict_ensemble.py",
    REPO_DIR / "requirements-inference.txt",
]
if any(not path.is_file() for path in required_code):
    raise RuntimeError(
        "指定した版には新しい推論コードがありません。更新ファイルを公開した後、対応するGIT_REFを指定してください。"
    )
print("Code commit:", CODE_COMMIT)

## 2. 推論用ライブラリを準備する

GPUを使う場合は、Colabのランタイム設定でGPUを選択してください。CPUでも実行できますが、TTAを含む3モデルの推論には時間がかかります。

Colabに入っているPyTorch・torchvisionの組を保持して、推論用の依存関係を追加します。学習環境の履歴であるルートの `requirements.txt` は、このセルでは使用しません。インストール後に再起動を求められた場合は、ランタイムを再起動して最初から実行してください。

In [ ]:
try:
    torch_version = metadata.version("torch")
    torchvision_version = metadata.version("torchvision")
except metadata.PackageNotFoundError as exc:
    raise RuntimeError("PyTorchとtorchvisionがあるColabランタイムを選択してください。") from exc

ENV_DIR = Path("/content/ecg_inference_environment")
ENV_DIR.mkdir(parents=True, exist_ok=True)
constraints_path = ENV_DIR / "colab_torch_constraints.txt"
constraints_path.write_text(
    f"torch=={torch_version}\ntorchvision=={torchvision_version}\n", encoding="utf-8"
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-inference.txt"),
     "-c", str(constraints_path)],
    check=True,
)
print("Preserved PyTorch / torchvision:", torch_version, "/", torchvision_version)

In [ ]:
import torch
import torchvision
from PIL import Image

runtime_versions = {"python": sys.version.split()[0]}
for package in ("torch", "torchvision", "timm", "transformers", "numpy", "Pillow", "safetensors"):
    runtime_versions[package] = metadata.version(package)
runtime_versions["cuda_available"] = torch.cuda.is_available()
runtime_versions["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print(json.dumps(runtime_versions, ensure_ascii=False, indent=2))

## 3. 重みと画像を指定する

推論用のファイル構成は次のとおりです。通常の学習checkpointを直接アップロードせず、`tools/export_inference_weights.py`で作成したファイルを使用してください。

- `weights/vit/model.safetensors` と `weights/vit/config.json`
- `weights/convnextv2/model.safetensors`
- `weights/mlpmixer/model.safetensors`

ViTの `preprocessor_config.json` は設定の記録として一緒に保管できます。元のデータセット評価コードに合わせ、推論では3モデルともImageNetのmean/stdで正規化します。ConvNeXtV2は `ckpt_0011.pth` の通常のモデル重みを使用し、SWAの重みは使用しません。

画像と重みはColabのファイルパネルから配置するか、必要な場合だけ次のセルでGoogle Driveを接続します。Drive内のファイルは入力として読み込み、結果は既定で `/content/ecg_inference_results` に保存します。

In [ ]:
USE_GOOGLE_DRIVE = False  # Driveから読み込む場合だけTrueに変更
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

In [ ]:
# "ensemble", "vit", "convnextv2", "mlpmixer" のいずれか
MODEL = "ensemble"

# Colabに配置したファイルの一般的なパス。手元の配置に合わせて変更してください。
WEIGHTS_DIR = Path("/content/weights")
IMAGE_PATH = Path("/content/data/example.png")
# Driveを使用する場合の例:
# WEIGHTS_DIR = Path("/content/drive/MyDrive/ecg_inference/weights")
# IMAGE_PATH = Path("/content/drive/MyDrive/ecg_inference/data/example.png")

OUTPUT_DIR = Path("/content/ecg_inference_results")
OUTPUT_JSON = OUTPUT_DIR / "prediction.json"

# 論文のTTA条件。TTA=1は変換なしのみ。
VIT_TTA = 24
VIT_MAGNITUDE = 1
CONVNEXTV2_TTA = 1
CONVNEXTV2_MAGNITUDE = 5  # TTA=1では使用されない
MLPMIXER_TTA = 32
MLPMIXER_MAGNITUDE = 3
ENSEMBLE_WEIGHTS = [1.0, 1.0, 1.0]  # ViT, ConvNeXtV2, MLP-Mixerの順

THRESHOLD = None  # None: ensembleはAMI>=0.5、単一モデルはargmax。数値で変更可能
SEED = 1234
DEVICE = "auto"  # "auto", "cpu", "cuda"
PRECISION = "float32"  # "float32" または "bfloat16"
INPUT_POLICY = "strict480"  # "resize" を明示すると480x480にリサイズ
BATCH_SIZE = 1
NUM_WORKERS = 4  # 元の評価設定。必要な場合は0に変更可能

## 4. 入力を確認して推論する

`strict480` は、元のデータセット評価と同じ **480×480画素** の入力を要求します。異なるサイズの場合は学習時と同じ前処理を確認してください。`resize` は明示的な代替設定であり、論文時の画像作成処理との一致を保証するものではありません。

TTAの乱数はseedで制御します。ただし、画像順・バッチサイズ・worker数・ライブラリや精度が異なる場合、当時の一括評価と完全には一致しないことがあります。

In [ ]:
supported_models = ("ensemble", "vit", "convnextv2", "mlpmixer")
if MODEL not in supported_models:
    raise ValueError(f"MODELは{supported_models}から選択してください。")
if not IMAGE_PATH.is_file():
    raise FileNotFoundError("IMAGE_PATHに入力画像を配置し、パスを指定してください。")

selected_models = ("vit", "convnextv2", "mlpmixer") if MODEL == "ensemble" else (MODEL,)
required_weights = [WEIGHTS_DIR / model / "model.safetensors" for model in selected_models]
if "vit" in selected_models:
    required_weights.append(WEIGHTS_DIR / "vit" / "config.json")
if any(not path.is_file() for path in required_weights):
    raise FileNotFoundError("選択したモデルの推論用重み・設定が不足しています。上記の構成を確認してください。")

with Image.open(IMAGE_PATH) as image:
    input_size = image.size
if INPUT_POLICY == "strict480" and input_size != (480, 480):
    raise ValueError(f"入力は{input_size}画素です。strict480には480x480の画像が必要です。")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Input dimensions:", input_size)
print("Selected model:", MODEL)

In [ ]:
command = [
    sys.executable,
    str(REPO_DIR / "examples" / f"predict_{MODEL}.py"),
    "--image", str(IMAGE_PATH),
    "--weights-dir", str(WEIGHTS_DIR),
    "--output", str(OUTPUT_JSON),
    "--vit-tta", str(VIT_TTA), "--vit-magnitude", str(VIT_MAGNITUDE),
    "--convnextv2-tta", str(CONVNEXTV2_TTA), "--convnextv2-magnitude", str(CONVNEXTV2_MAGNITUDE),
    "--mlpmixer-tta", str(MLPMIXER_TTA), "--mlpmixer-magnitude", str(MLPMIXER_MAGNITUDE),
    "--ensemble-weights", *[str(value) for value in ENSEMBLE_WEIGHTS],
    "--seed", str(SEED),
    "--device", DEVICE,
    "--precision", PRECISION,
    "--input-policy", INPUT_POLICY,
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
]
if THRESHOLD is not None:
    command.extend(["--threshold", str(THRESHOLD)])
subprocess.run(command, cwd=REPO_DIR, check=True)
result = json.loads(OUTPUT_JSON.read_text(encoding="utf-8"))
print("Inference completed.")

## 5. 結果を確認する

クラス順は **0 = tako、1 = AMI** です。既定の判定は、アンサンブルでは `prob_AMI >= 0.5`、単一モデルではargmaxです（確率が同率の場合はそれぞれAMI／tako）。`THRESHOLD`に数値を指定すると、どちらもその閾値で判定します。以下にはモデル出力を表示します。正解ラベルとの比較を行っていないため、精度・感度・特異度を示すものではありません。

JSONには実際の実行条件、重みの識別情報、logitsと確率が保存されます。Colabの作業領域は永続保存ではないため、必要な結果を保存してください。

In [ ]:
for prediction in result["predictions"]:
    summary = {
        "sample_id": prediction.get("sample_id"),
        "pred_id": prediction["pred_id"],
        "pred_label": prediction["pred_label"],
        "logits": prediction["logits"],
        "probabilities": prediction["probabilities"],
        "models": prediction.get("models", {}),
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))

# コードと実行環境を追跡する補助記録（入力の絶対パスは記録しない）
(OUTPUT_DIR / "colab_run.json").write_text(
    json.dumps({"code_commit": CODE_COMMIT, "runtime": runtime_versions}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

In [ ]:
# 必要な場合だけTrueに変更してJSONをダウンロード
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(OUTPUT_JSON))
    files.download(str(OUTPUT_DIR / "colab_run.json"))

## 次の確認

論文の既存予測との照合には、同じ重み、同じ入力画像、同じ前処理を使用します。まずTTAなしでlogitsを照合し、次に指定したTTA、最後にlogit平均アンサンブルを確認します。

このnotebookの配布版には実患者画像・実行結果を保存していません。Colabでの実行確認と、論文時の予測との一致確認は、必要な重み・画像を用意した環境で行ってください。